In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
!pip install langchain langchain-experimental langchain-community langchain-openai openai chromadb pypdf sentence_transformers gradio langchain-together pypdf

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
from pypdf import PdfReader

reader = PdfReader("Moni_Hazarika_Resume.pdf")

In [ ]:
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
uploaded = files.upload()

In [ ]:
with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

print(summary)

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
import getpass
import os

from langchain_community.utilities import WikipediaAPIWrapper, SerpAPIWrapper
from langchain_community.tools import WikipediaQueryRun
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


# =====================================================
# Step 0: API Keys
# =====================================================
os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Enter OpenRouter API Key: ")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

In [ ]:
llm = ChatOpenAI(
    model="openai/gpt-4o-mini",
    temperature=0.3,
    max_tokens=500,
    openai_api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content=system_prompt),
    HumanMessage(content="Tell me about AI agents.")
]

response = llm.invoke(messages)

print(response.content)


In [ ]:
history = [
    {"role": "user", "content": "Hi"},
    {"role": "assistant", "content": "Hello!"}
]

In [ ]:
from langchain_core.messages import (
    SystemMessage,
    HumanMessage,
    AIMessage,
)


def chat(message, history):

    messages = [SystemMessage(content=system_prompt)]

    # Add previous conversation
    for item in history:
        if item["role"] == "user":
            messages.append(
                HumanMessage(content=item["content"])
            )
        elif item["role"] == "assistant":
            messages.append(
                AIMessage(content=item["content"])
            )

    # Add latest user message
    messages.append(
        HumanMessage(content=message)
    )

    response = llm.invoke(messages)

    return response.content

OpenAI

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1"
)

In [ ]:
from IPython.display import Markdown, display

messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": "Tell me about AI agents."
    }
]

response = client.chat.completions.create(
    model="openai/gpt-4o-mini",
    messages=messages,
    temperature=0.3,
    max_tokens=500
)

display(
    Markdown(
        response.choices[0].message.content
    )
)

In [ ]:
def chat(message, history):

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages,
        temperature=0.3,
        max_tokens=500
    )

    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

GRADIO

In [ ]:
import gradio as gr


def chat(message, history):

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages,
        temperature=0.3,
        max_tokens=500
    )

    return response.choices[0].message.content


gr.ChatInterface(chat).launch()

TOOLS

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

In [ ]:
import json


def chat(message, history):

    messages = (
        [{"role": "system", "content": system_prompt}]
        + history
        + [{"role": "user", "content": message}]
    )

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages,
        tools=tools,
        temperature=0.3,
        max_tokens=500
    )

    # Did the model decide to call a tool?
    if response.choices[0].finish_reason == "tool_calls":

        assistant_message = response.choices[0].message

        # Add the assistant's tool call message
        messages.append(assistant_message)

        # Handle all tool calls
        for tool_call in assistant_message.tool_calls:

            if tool_call.function.name == "record_email":

                arguments = json.loads(
                    tool_call.function.arguments
                )

                email = arguments.get("email")

                # Execute the tool
                record_email_tool(email)

                # Send the tool's result back
                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": "Email recorded successfully."
                    }
                )

        # Ask the model for its final response
        response = client.chat.completions.create(
            model="openai/gpt-4o-mini",
            messages=messages,
            tools=tools,
            temperature=0.3,
            max_tokens=500
        )

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

EXERCISE

Step 1: Create an evaluator

After your main response is generated, call the LLM again to judge it.

In [ ]:
def evaluate_response(response_text):

    evaluation_prompt = f"""
You are a compliance reviewer.

Determine whether this response is strictly related to professional work,
business, technology, engineering, AI, software, data, cloud, leadership,
or career topics.

Response:
{response_text}

Reply ONLY with:
PASS
or
FAIL
"""

    evaluation = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a strict evaluator."
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ],
        temperature=0
    )

    return evaluation.choices[0].message.content.strip()

Step 2: Add optimizer

If evaluation fails, ask another LLM call to rewrite.

In [ ]:
def optimize_response(response_text):

    optimization_prompt = f"""
Rewrite the following response so it is strictly related to
professional work and business topics.

Response:
{response_text}
"""

    optimized = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "You are a business communication expert."
            },
            {
                "role": "user",
                "content": optimization_prompt
            }
        ],
        temperature=0.3
    )

    return optimized.choices[0].message.content

Step 3: Add to chat flow

After tool handling and before returning:

In [ ]:
def evaluate_response(response_text):

    prompt = f"""
Determine whether the following response is strictly related to work.

Reply ONLY with one word:

PASS
or
FAIL

Response:
{response_text}
"""

    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=[
            {
                "role":"system",
                "content":"You are a strict evaluator."
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [ ]:
evaluation = evaluate_response(reply)

if evaluation == "FAIL":
    reply = optimize_response(reply)

Applying it to your business

For the second part of the exercise, imagine you're building an assistant for Swimlane's security automation platform.

System Prompt

In [ ]:
system_prompt = """
You are a Swimlane AI Assistant.

You help users with:

- Security automation
- SOAR
- Incident response
- Case management
- Threat intelligence
- Elasticsearch
- MongoDB
- Kubernetes
- Cloud infrastructure
- AI agents

If a user wants to be contacted,
ask for their email address.

When they provide an email address,
call the record_email_tool.

Do not answer unrelated topics such as:
- Dating
- Movies
- Politics
- Sports
- Personal advice
"""

In [ ]:
gr.ChatInterface(chat).launch()

                   USER
                     |
                     |
                 chat()
                     |
                     |
               Generator LLM
                     |
                     |
              Tool required?
                /         \
              Yes          No
               |            |
          execute tool      |
               |            |
               ----------------
                       |
                    reply
                       |
                 Evaluator LLM
                       |
                 PASS / FAIL
                  /       \
               PASS       FAIL
                 |          |
              return    Optimizer LLM
                            |
                         rewrite
                            |
                         return